# BRM on the bike dataset (regression)

This notebook demonstrates Blockwise Reduced Modeling (BRM) on the **Capital Bikeshare** hourly-demand dataset. Because `bike` is otherwise complete, we induce a realistic blockwise missing pattern with `simulate_blockwise_missing()` to show BRM in its element.

Method reference: Srinivasan, Currim, and Ram (2025), *A Reduced Modeling Approach for Making Predictions With Incomplete Data Having Blockwise Missing Patterns*, INFORMS Journal on Data Science.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

from blockwise import BRM, simulate_blockwise_missing, choose_num_blocks, datasets

rng = np.random.default_rng(1234)

## Load the bike data

In [ ]:
bike = datasets.load_bike()
print(bike.shape)
bike.head()

## Induce a blockwise missing pattern

Two joint-masked column groups — simulating two independent data sources failing on different subsets of rows — plus a small per-column noise rate.

In [ ]:
bike_miss = simulate_blockwise_missing(
    bike,
    blocks=[
        ["windspeed", "hum", "weekday"],
        ["hr", "temp", "weathersit"],
    ],
    prop_missing=0.30,
    noise=0.05,
)

(bike_miss.isna().mean() * 100).round(1)

## Train / test split

In [ ]:
# Categorical columns: convert to one-hot for the sklearn estimators below.
bike_encoded = pd.get_dummies(bike_miss, columns=[c for c in ["season", "weekday", "weathersit"] if c in bike_miss.columns], drop_first=False)

X = bike_encoded.drop(columns=["cnt"])
y = bike_encoded["cnt"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1234)
print(X_train.shape, X_test.shape)

## Let BRM choose the number of blocks, then fit with a linear model

In [ ]:
k_info = choose_num_blocks(X_train, k_max=10)
print("elbow k:", k_info["n_blocks"])
print("curve :", np.round(k_info["missing_curve"], 3))

brm_lm = BRM(estimator=LinearRegression(), n_blocks=k_info["n_blocks"]).fit(X_train, y_train)
brm_lm

In [ ]:
pred_lm = brm_lm.predict(X_test)
rmse_lm = float(np.sqrt(np.mean((y_test - pred_lm) ** 2)))
print(f"BRM (LinearRegression) RMSE: {rmse_lm:.2f}")

## Same pipeline with a gradient-boosted tree learner

BRM is learner-agnostic — any sklearn estimator with `fit/predict` plugs in. Each block gets an independent clone.

In [ ]:
brm_gb = BRM(
    estimator=GradientBoostingRegressor(random_state=0, n_estimators=200),
    n_blocks=k_info["n_blocks"],
).fit(X_train, y_train)

pred_gb = brm_gb.predict(X_test)
rmse_gb = float(np.sqrt(np.mean((y_test - pred_gb) ** 2)))
print(f"BRM (GradientBoostingRegressor) RMSE: {rmse_gb:.2f}")

## Listwise-deletion baseline

The conventional alternative is to drop any row with a missing value and fit a single model on what remains. BRM keeps all the training rows (split across per-pattern subsets); listwise deletion throws them away.

In [ ]:
train_df = pd.concat([X_train, pd.Series(y_train, name="cnt", index=X_train.index)], axis=1)
complete = train_df.dropna()
print(f"rows: BRM used {len(X_train)}, listwise used {len(complete)}")

lw = LinearRegression().fit(complete.drop(columns=["cnt"]), complete["cnt"])

# Fill the test set with training-column means to make the baseline predict at all.
X_test_imp = X_test.copy()
for c in X_test_imp.columns:
    if X_test_imp[c].isna().any():
        X_test_imp[c] = X_test_imp[c].fillna(complete[c].mean() if c in complete.columns else 0)

pred_lw = lw.predict(X_test_imp)
rmse_lw = float(np.sqrt(np.mean((y_test - pred_lw) ** 2)))
print(f"Listwise-deletion LinearRegression RMSE: {rmse_lw:.2f}")

## Citation

If you use BRM, please cite:

> Srinivasan, K., Currim, F., and Ram, S. (2025). *A Reduced Modeling Approach for Making Predictions With Incomplete Data Having Blockwise Missing Patterns.* INFORMS Journal on Data Science.